In [0]:
from pyspark.sql.functions import col, abs, countDistinct, startswith, sum,count
from pyspark.sql.types import *

In [0]:
df = spark.table("workspace.retail_pipeline_test.silver_sales_v2")

In [0]:
revenue_df=df.withColumn("Revenue",col("quantity")*col("UnitPrice"))
display(revenue_df)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,source_file,ingestion_timestamp,batch_id,source_system,Revenue
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.55,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,15.30
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,20.34
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.75,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,22.00
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,20.34
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,20.34
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.65,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,15.30
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,4.25,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,25.50
536366,22633,HAND WARMER UNION JACK,6,2010-12-01T08:28:00.000Z,1.85,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,11.10
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01T08:28:00.000Z,1.85,17850,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,11.10
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T08:34:00.000Z,1.69,13047,United Kingdom,/Volumes/workspace/retail_pipeline_test/retail_raw_files/online_retail.csv,2026-10-13T20:31:01.907Z,initial_load_001,online_retail_uci,54.08


In [0]:
revenue_df.filter(col("Quantity") <0).agg(abs(sum("Revenue"))).show()

+-----------------+
|abs(sum(Revenue))|
+-----------------+
|        896812.49|
+-----------------+



In [0]:
revenue_df.agg(countDistinct("InvoiceNo")).show()

+-------------------------+
|count(DISTINCT InvoiceNo)|
+-------------------------+
|                    25900|
+-------------------------+



In [0]:
revenue_df.filter(
    (~col("InvoiceNo").startswith("C"))
    &
    (col("Quantity") > 0)
    &
    (~col("InvoiceNo").startswith("A"))
).agg(
    sum("Revenue")
).show()

+------------+
|sum(Revenue)|
+------------+
| 10655622.48|
+------------+



In [0]:
revenue_df.filter(col("UnitPrice")< 0).show()

+---------+---------+---------------+--------+-------------------+---------+----------+--------------+--------------------+--------------------+----------------+-----------------+---------+
|InvoiceNo|StockCode|    Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|         source_file| ingestion_timestamp|        batch_id|    source_system|  Revenue|
+---------+---------+---------------+--------+-------------------+---------+----------+--------------+--------------------+--------------------+----------------+-----------------+---------+
|  A563186|        B|Adjust bad debt|       1|2011-08-12 14:51:00|-11062.06|        -1|United Kingdom|/Volumes/workspac...|2026-10-13 20:31:...|initial_load_001|online_retail_uci|-11062.06|
|  A563187|        B|Adjust bad debt|       1|2011-08-12 14:52:00|-11062.06|        -1|United Kingdom|/Volumes/workspac...|2026-10-13 20:31:...|initial_load_001|online_retail_uci|-11062.06|
+---------+---------+---------------+--------+----

In [0]:
revenue_df.filter(
    (col("Quantity") > 0)
    &
    (
        col("InvoiceNo").startswith("A")
        |
        col("InvoiceNo").startswith("C")
    )
).select(
    "InvoiceNo",
    "Description",
    "Quantity",
    "UnitPrice",
    "Revenue"
).show(30, truncate=False)

+---------+---------------+--------+---------+---------+
|InvoiceNo|Description    |Quantity|UnitPrice|Revenue  |
+---------+---------------+--------+---------+---------+
|A563185  |Adjust bad debt|1       |11062.06 |11062.06 |
|A563186  |Adjust bad debt|1       |-11062.06|-11062.06|
|A563187  |Adjust bad debt|1       |-11062.06|-11062.06|
+---------+---------------+--------+---------+---------+



In [0]:
retail_df = revenue_df.filter(
    ~col("InvoiceNo").startswith("A")
)

In [0]:
retail_df.groupBy(
    col("InvoiceNo").substr(1, 1).alias("InvoicePrefix")
).agg(
    count("*").alias("Rows"),
    sum("Revenue").alias("TotalRevenue")
).orderBy("InvoicePrefix").show()

+-------------+------+------------+
|InvoicePrefix|  Rows|TotalRevenue|
+-------------+------+------------+
|            5|532618| 10655622.48|
|            C|  9288|  -896812.49|
+-------------+------+------------+



In [0]:
sales_df = retail_df.filter(
    ~col("InvoiceNo").startswith("C")
)

returns_df = retail_df.filter(
    col("InvoiceNo").startswith("C")
)

In [0]:
import pyspark.sql.functions as F

gross_df = sales_df.agg(
    F.sum("Revenue").alias("GrossRevenue")
)

returns_kpi_df = returns_df.agg(
    F.abs(F.sum("Revenue")).alias("ReturnsValue")
)

net_df = retail_df.agg(
    F.sum("Revenue").alias("NetRevenue")
)

In [0]:
gross_df.show()
returns_kpi_df.show()
net_df.show()

+------------+
|GrossRevenue|
+------------+
| 10655622.48|
+------------+

+------------+
|ReturnsValue|
+------------+
|   896812.49|
+------------+

+----------+
|NetRevenue|
+----------+
|9758809.99|
+----------+



In [0]:
customers_df = retail_df.filter(
    F.col("CustomerID") != "-1"
).agg(
    F.countDistinct("CustomerID").alias("TotalCustomers")
)

orders_df = sales_df.filter(
    F.col("Quantity") > 0
).agg(
    F.countDistinct("InvoiceNo").alias("TotalOrders")
)

customers_df.show()
orders_df.show()

+--------------+
|TotalCustomers|
+--------------+
|          4372|
+--------------+

+-----------+
|TotalOrders|
+-----------+
|      20725|
+-----------+



In [0]:
aov_df = net_df.crossJoin(orders_df).select(
    F.round(
        F.col("NetRevenue") / F.col("TotalOrders"), 2
    ).alias("AOV")
)

aov_df.show()

+------+
|   AOV|
+------+
|470.87|
+------+



In [0]:
gold_kpis_df = (
    gross_df
    .crossJoin(returns_kpi_df)
    .crossJoin(net_df)
    .crossJoin(customers_df)
    .crossJoin(orders_df)
    .crossJoin(aov_df)
)

gold_kpis_df.show()

+------------+------------+----------+--------------+-----------+------+
|GrossRevenue|ReturnsValue|NetRevenue|TotalCustomers|TotalOrders|   AOV|
+------------+------------+----------+--------------+-----------+------+
| 10655622.48|   896812.49|9758809.99|          4372|      20725|470.87|
+------------+------------+----------+--------------+-----------+------+



In [0]:
assert gold_kpis_df.count() == 1, "Expected exactly one KPI row"

gold_kpis_df.printSchema()

root
 |-- GrossRevenue: decimal(31,2) (nullable = true)
 |-- ReturnsValue: decimal(31,2) (nullable = true)
 |-- NetRevenue: decimal(31,2) (nullable = true)
 |-- TotalCustomers: long (nullable = false)
 |-- TotalOrders: long (nullable = false)
 |-- AOV: decimal(32,2) (nullable = true)



In [0]:
gold_kpis_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.retail_pipeline_test.gold_kpis_v2"
    )

In [0]:
saved_kpis_df = spark.table(
    "workspace.retail_pipeline_test.gold_kpis_v2"
)

saved_kpis_df.show()

+------------+------------+----------+--------------+-----------+------+
|GrossRevenue|ReturnsValue|NetRevenue|TotalCustomers|TotalOrders|   AOV|
+------------+------------+----------+--------------+-----------+------+
| 10655622.48|   896812.49|9758809.99|          4372|      20725|470.87|
+------------+------------+----------+--------------+-----------+------+



In [0]:
import pyspark.sql.functions as F

row = saved_kpis_df.first()

assert all(value is not None for value in row), \
    "KPI contains NULL values"

assert row["GrossRevenue"] - row["ReturnsValue"] == row["NetRevenue"], \
    "Revenue reconciliation failed"

assert row["TotalOrders"] > 0, \
    "TotalOrders must be positive"

print("All KPI validation tests passed!")

All KPI validation tests passed!
